# 事前学習と最小言語モデル

大規模言語モデルの事前学習は、文脈から次トークン分布を出す関数を鍛える作業である。入力列をトークン ID に変換し、直前までの文脈から次の ID を予測し、外れ方を損失として重みに戻す。

小さな文字レベルのモデルでも、データ化、teacher forcing、交差エントロピー、勾配更新、サンプリング、因果マスクの関係は同じ形で現れる。

In [ ]:
import math
import random

random.seed(42)

corpus = [
    'models learn patterns from data',
    'tokens become vectors before prediction',
    'attention mixes earlier context',
    'pretraining predicts the next token',
    'loss sends a correction through weights',
    'sampling turns probabilities into text',
]

print('documents:', len(corpus))
print('characters:', sum(len(text) for text in corpus))

言語モデルは列全体の確率を次トークン確率の積として扱う。

`P(x1, ..., xT) = P(x1) P(x2 | x1) ... P(xT | x1, ..., xT-1)`

学習時は正しい過去列を入力し、各位置で次トークンを当てる。これを teacher forcing と呼ぶ。

In [ ]:
BOS = '<bos>'
chars = sorted(set(''.join(corpus)))
itos = [BOS] + chars
stoi = {token: idx for idx, token in enumerate(itos)}
vocab_size = len(itos)

def encode(text):
    return [stoi[ch] for ch in text]


def decode(ids):
    return ''.join(itos[i] for i in ids if i != 0)

encoded_docs = [[0] + encode(text) + [0] for text in corpus]

print('vocab size:', vocab_size)
print('first ids:', encoded_docs[0][:18])
print('round trip:', decode(encoded_docs[0]))

文脈長を固定すると、各訓練例は「直前 `block_size` 個の ID」と「次の ID」の組になる。左側を `BOS` で埋めれば、文頭付近も同じ形で扱える。

In [ ]:
block_size = 12

contexts = []
targets = []
for ids in encoded_docs:
    for pos in range(1, len(ids)):
        left = ids[max(0, pos - block_size):pos]
        padded = [0] * (block_size - len(left)) + left
        contexts.append(padded)
        targets.append(ids[pos])

print('training pairs:', len(contexts))
print('context ids:', contexts[12])
print('target:', targets[12], itos[targets[12]])

最初の基準として bigram モデルを置く。直前 1 トークンだけを見て次トークン分布を作るため、長い依存は扱えないが、尤度と生成の流れを確認しやすい。

In [ ]:
counts = [[1.0 for _ in range(vocab_size)] for _ in range(vocab_size)]
for ids in encoded_docs:
    for prev_id, next_id in zip(ids[:-1], ids[1:]):
        counts[prev_id][next_id] += 1.0

bigram_probs = []
for row in counts:
    total = sum(row)
    bigram_probs.append([value / total for value in row])

loss_terms = []
for ctx, target in zip(contexts, targets):
    loss_terms.append(-math.log(bigram_probs[ctx[-1]][target]))
bigram_nll = sum(loss_terms) / len(loss_terms)

after_t = [(itos[i], round(p, 3)) for i, p in enumerate(bigram_probs[stoi['t']])]
after_t = sorted(after_t, key=lambda item: item[1], reverse=True)[:6]

print('bigram NLL:', round(bigram_nll, 3))
print('after "t":', after_t)

次に、埋め込み、位置情報、文脈の平均、線形出力層を持つ小さなモデルを学習する。Transformer ほど強くないが、次トークン事前学習で必要な主要部品はそろっている。

In [ ]:
d_model = 24


def zeros(rows, cols):
    return [[0.0 for _ in range(cols)] for _ in range(rows)]


def rand_matrix(rows, cols, scale):
    return [[random.gauss(0.0, scale) for _ in range(cols)] for _ in range(rows)]


E = rand_matrix(vocab_size, d_model, 0.08)
P = rand_matrix(block_size, d_model, 0.02)
W = rand_matrix(d_model, vocab_size, 0.08)
b = [0.0 for _ in range(vocab_size)]


def softmax(logits):
    base = max(logits)
    exp_values = [math.exp(value - base) for value in logits]
    total = sum(exp_values)
    return [value / total for value in exp_values]


def forward_one(context):
    z = []
    for pos, token_id in enumerate(context):
        z.append([E[token_id][j] + P[pos][j] for j in range(d_model)])
    h = [sum(z[pos][j] for pos in range(block_size)) / block_size for j in range(d_model)]
    logits = []
    for out_id in range(vocab_size):
        value = b[out_id]
        for j in range(d_model):
            value += h[j] * W[j][out_id]
        logits.append(value)
    return z, h, logits, softmax(logits)


def batch_loss(batch_indices):
    total = 0.0
    for idx in batch_indices:
        _, _, _, probs = forward_one(contexts[idx])
        total += -math.log(probs[targets[idx]] + 1e-12)
    return total / len(batch_indices)

all_indices = list(range(len(contexts)))
print('initial NLL:', round(batch_loss(all_indices), 3))

交差エントロピーの勾配は、正解 ID の確率を上げ、他の ID の確率を下げる向きになる。各ミニバッチでその向きを平均し、埋め込み、位置ベクトル、出力層を同時に更新する。

In [ ]:
def loss_and_grads(batch_indices):
    dE = zeros(vocab_size, d_model)
    dP = zeros(block_size, d_model)
    dW = zeros(d_model, vocab_size)
    db = [0.0 for _ in range(vocab_size)]
    total_loss = 0.0
    scale = 1.0 / len(batch_indices)

    for idx in batch_indices:
        context = contexts[idx]
        target = targets[idx]
        z, h, logits, probs = forward_one(context)
        total_loss += -math.log(probs[target] + 1e-12)

        dlogits = probs[:]
        dlogits[target] -= 1.0
        dlogits = [value * scale for value in dlogits]

        for j in range(d_model):
            for out_id in range(vocab_size):
                dW[j][out_id] += h[j] * dlogits[out_id]
        for out_id in range(vocab_size):
            db[out_id] += dlogits[out_id]

        dh = [0.0 for _ in range(d_model)]
        for j in range(d_model):
            dh[j] = sum(W[j][out_id] * dlogits[out_id] for out_id in range(vocab_size))

        for pos, token_id in enumerate(context):
            for j in range(d_model):
                grad = dh[j] / block_size
                dE[token_id][j] += grad
                dP[pos][j] += grad

    return total_loss * scale, dE, dP, dW, db


def apply_grads(grads, lr):
    dE, dP, dW, db = grads
    for token_id in range(vocab_size):
        for j in range(d_model):
            E[token_id][j] -= lr * dE[token_id][j]
    for pos in range(block_size):
        for j in range(d_model):
            P[pos][j] -= lr * dP[pos][j]
    for j in range(d_model):
        for out_id in range(vocab_size):
            W[j][out_id] -= lr * dW[j][out_id]
    for out_id in range(vocab_size):
        b[out_id] -= lr * db[out_id]

loss_history = []
for step in range(520):
    batch = [random.randrange(len(contexts)) for _ in range(48)]
    loss, dE, dP, dW, db = loss_and_grads(batch)
    loss_history.append(loss)
    apply_grads((dE, dP, dW, db), lr=0.55)

print('final NLL:', round(batch_loss(all_indices), 3))
print('first losses:', [round(v, 3) for v in loss_history[:5]])
print('last losses:', [round(v, 3) for v in loss_history[-5:]])

推論では正解の過去列を使えない。生成済み ID を文脈に戻し、温度で分布の鋭さを調整して次トークンをサンプリングする。

In [ ]:
def sample_from_probs(probs):
    r = random.random()
    running = 0.0
    for idx, prob in enumerate(probs):
        running += prob
        if r <= running:
            return idx
    return len(probs) - 1


def sample_next(context, temperature=0.8):
    _, _, logits, _ = forward_one(context[-block_size:])
    scaled = [value / temperature for value in logits]
    return sample_from_probs(softmax(scaled))


def generate(max_new_tokens=60, temperature=0.8):
    ids = [0] * block_size
    out = []
    for _ in range(max_new_tokens):
        next_id = sample_next(ids, temperature)
        if next_id == 0 and out:
            break
        ids.append(next_id)
        out.append(next_id)
    return decode(out)

for temp in [0.45, 0.8, 1.2]:
    print(temp, '->', generate(temperature=temp))

Transformer の自己注意は、各位置の query が過去位置の key と照合し、value の重み付き和を作る。未来位置を見ないようにするため、因果マスクで右上を消す。

In [ ]:
def dot(a, b):
    return sum(x * y for x, y in zip(a, b))


def attention_weights(token_ids):
    z = []
    for pos, token_id in enumerate(token_ids[-block_size:]):
        z.append([E[token_id][j] + P[pos][j] for j in range(d_model)])
    rows = []
    for q_pos in range(block_size):
        scores = []
        for k_pos in range(block_size):
            if k_pos > q_pos:
                scores.append(-1e9)
            else:
                scores.append(dot(z[q_pos], z[k_pos]) / math.sqrt(d_model))
        rows.append(softmax(scores))
    return rows

probe = [0] * (block_size - 6) + encode('tokens')
weights = attention_weights(probe)

for row in weights[-6:]:
    print([round(value, 2) for value in row[-6:]])
print('row sums:', [round(sum(row), 3) for row in weights[-6:]])

事前学習の中心は、次トークン確率を出す関数を大量の列で更新し続けることにある。bigram は直前だけを使うため弱く、埋め込みモデルは文脈をベクトルへ集約できる。Transformer はさらに自己注意で、位置ごとに参照すべき過去を変えられる。